# MLM vs Tukey diff — sweep across all figure directories

Walks every subdirectory under a figures save dir, finds every `*_tukey.csv` / `*_mlm.csv` pair, and reports which pairwise comparisons changed significance between the two methods.

**Output:** a single summary CSV with one row per (file_pair, comparison) that flipped, plus a printed digest. Useful for the response-to-reviewers — answers "how many results actually changed?"

**Prerequisites:** you must have already regenerated figures with both `use_mlm=False` (writes `_tukey.csv`) and `use_mlm=True` (writes `_mlm.csv`). Existing `_tukey.csv` files from prior runs work as the baseline.

In [25]:
import os
import glob
import pandas as pd
import numpy as np

from visual_behavior.data_access import loading as loading

## Configure the sweep

`figures_root` should point at the directory whose subfolders contain the saved CSVs. By default we look at `platform_paper_figures_final/PaperRevision_Neuron/` — change as needed.

In [26]:
loading.get_platform_analysis_cache_dir()

'/Users/marinag/Library/CloudStorage/Dropbox/JupyterNotebooks/FinalDataAssetsforCodeOcean/'

In [27]:
# figures_root = os.path.join(
#     loading.get_platform_analysis_cache_dir(),
#     'platform_paper_figures_final', 'PaperRevision_Neuron'
# )

figures_root = r'/Users/marinag/Library/CloudStorage/Dropbox/JupyterNotebooks/Figures'

print('Sweeping under:', figures_root)
assert os.path.exists(figures_root), f'figures_root does not exist: {figures_root}'

# Where to write the summary
output_csv = os.path.join(figures_root, 'mlm_vs_tukey_diff_summary.csv')

Sweeping under: /Users/marinag/Library/CloudStorage/Dropbox/JupyterNotebooks/Figures


## Find matching CSV pairs

Each `_tukey.csv` has a corresponding `_mlm.csv` if both modes were run. The stem (everything before the suffix) is the join key.

In [28]:
tukey_files = sorted(glob.glob(os.path.join(figures_root, '**', '*_tukey.csv'), recursive=True))
mlm_files = sorted(glob.glob(os.path.join(figures_root, '**', '*_mlm.csv'), recursive=True))
print(f'Found {len(tukey_files)} _tukey.csv files')
print(f'Found {len(mlm_files)} _mlm.csv files')

# Index by stem (path without the suffix)
def stem(path, suffix):
    return path[:-len(suffix)]

tukey_by_stem = {stem(p, '_tukey.csv'): p for p in tukey_files}
mlm_by_stem = {stem(p, '_mlm.csv'): p for p in mlm_files}

matched_stems = sorted(set(tukey_by_stem) & set(mlm_by_stem))
unmatched_tukey = sorted(set(tukey_by_stem) - set(mlm_by_stem))
unmatched_mlm = sorted(set(mlm_by_stem) - set(tukey_by_stem))

print(f'\n{len(matched_stems)} matched pairs ready for comparison')
if unmatched_tukey:
    print(f'{len(unmatched_tukey)} tukey files without an mlm counterpart (skipping)')
if unmatched_mlm:
    print(f'{len(unmatched_mlm)} mlm files without a tukey counterpart (skipping)')

Found 160 _tukey.csv files
Found 71 _mlm.csv files

53 matched pairs ready for comparison
107 tukey files without an mlm counterpart (skipping)
18 mlm files without a tukey counterpart (skipping)


In [29]:
#### List mlm files without a Tukey counterpart
if unmatched_mlm:
    print('\nMLM files without a Tukey counterpart:')
    for path in unmatched_mlm:
        print(f'  {path[60:]}')


MLM files without a Tukey counterpart:
  /Figures/platform_paper_figures/figure_2/behavior_metrics/max_dprime_stats
  /Figures/platform_paper_figures/figure_2/behavior_metrics/mean_hit_rate_stats
  /Figures/platform_paper_figures/figure_2/behavior_metrics/mean_hit_rate_uncorrected_stats
  /Figures/platform_paper_figures/figure_3_supplemental/metric_distributions/experience_modulation_heatmap_area_and_depth_by_cell_type_changes_anova
  /Figures/platform_paper_figures/figure_3_supplemental/metric_distributions/experience_modulation_heatmap_area_and_depth_by_cell_type_images_anova
  /Figures/platform_paper_figures/figure_3_supplemental/metric_distributions/experience_modulation_heatmap_area_and_depth_by_cell_type_omissions_anova
  /Figures/platform_paper_figures/figure_3_supplemental/response_metrics/bidirectional_metric_heatmap_area_and_depth_change_modulation_index_anova
  /Figures/platform_paper_figures/figure_3_supplemental/response_metrics/change_modulation_index_experience_levelyax

In [30]:
#### List tukey files without an mlm counterpart
if unmatched_tukey:
    print('\nTukey files without an MLM counterpart:')
    for path in unmatched_tukey:
        print(f'  {path[60:]}')


Tukey files without an MLM counterpart:
  /Figures/platform_paper_figures/figure_2_supplemental/behavior_metrics/fraction_engaged_stats_fraction_engaged
  /Figures/platform_paper_figures/figure_2_supplemental/behavior_metrics/max_dprime_stats_sdk_platform_experiments_F_N
  /Figures/platform_paper_figures/figure_2_supplemental/behavior_metrics/response_latency_mean_stats_response_latency
  /Figures/platform_paper_figures/figure_3/metric_distributions/changes_events_mean_response_changes_experience_level_boxplot
  /Figures/platform_paper_figures/figure_3/metric_distributions/changes_events_mean_response_changes_experience_level_no_cell_type
  /Figures/platform_paper_figures/figure_3/metric_distributions/changes_events_mean_response_changes_experience_level_pointplot
  /Figures/platform_paper_figures/figure_3/metric_distributions/changes_events_mean_response_changes_experience_level_violinplot
  /Figures/platform_paper_figures/figure_3/metric_distributions/changes_filtered_events_mean_re

## Diff each pair

For each pair, line up rows by `(group1, group2)` plus any extra grouping cols present in both tables (`cell_type`, `comparison`, `condition`, `data_subset`, etc.) and compute which rows flipped `reject`.

In [31]:
# Optional grouping columns to use as join keys if present in both tables.
# These let us distinguish e.g. Excitatory vs Sst vs Vip rows within the same file.
OPTIONAL_KEYS = ['cell_type', 'comparison', 'condition', 'data_subset', 'metric']

def build_join_keys(tukey_df, mlm_df):
    base = ['group1', 'group2']
    extras = [c for c in OPTIONAL_KEYS if c in tukey_df.columns and c in mlm_df.columns]
    return base + extras

def diff_pair(tukey_path, mlm_path):
    try:
        tdf = pd.read_csv(tukey_path)
        mdf = pd.read_csv(mlm_path)
    except Exception as e:
        return None, f'read error: {e}'

    if 'reject' not in tdf.columns or 'reject' not in mdf.columns:
        return None, 'missing reject column'

    keys = build_join_keys(tdf, mdf)
    # cast keys to str to avoid dtype-mismatch join misses
    for df in (tdf, mdf):
        for k in keys:
            df[k] = df[k].astype(str)

    merged = tdf[keys + ['reject']].merge(
        mdf[keys + ['reject', 'pvalue_adj', 'omnibus_pvalue', 'icc', 'model_type', 'n_groups', 'n_observations']],
        on=keys, how='outer', suffixes=('_tukey', '_mlm'), indicator=True
    )
    return merged, None

rows = []
errors = []
for stem_path in matched_stems:
    tpath = tukey_by_stem[stem_path]
    mpath = mlm_by_stem[stem_path]
    merged, err = diff_pair(tpath, mpath)
    if err:
        errors.append((stem_path, err))
        continue

    rel = os.path.relpath(stem_path, figures_root)
    for _, row in merged.iterrows():
        old = row.get('reject_tukey')
        new = row.get('reject_mlm')
        if pd.isnull(old) or pd.isnull(new):
            status = 'only_in_' + ('mlm' if pd.isnull(old) else 'tukey')
            changed = None
        else:
            old_b = str(old).lower() in ('true', '1', '1.0')
            new_b = str(new).lower() in ('true', '1', '1.0')
            changed = old_b != new_b
            status = 'flipped' if changed else 'same'

        rows.append({
            'figure_stem': rel,
            'group1': row.get('group1'),
            'group2': row.get('group2'),
            **{k: row.get(k) for k in OPTIONAL_KEYS if k in row.index},
            'tukey_reject': old,
            'mlm_reject': new,
            'changed': changed,
            'status': status,
            'mlm_pvalue_adj': row.get('pvalue_adj'),
            'mlm_omnibus_pvalue': row.get('omnibus_pvalue'),
            'mlm_icc': row.get('icc'),
            'mlm_model_type': row.get('model_type'),
            'mlm_n_groups': row.get('n_groups'),
            'mlm_n_observations': row.get('n_observations'),
        })

diff_df = pd.DataFrame(rows)
print(f'\nProcessed {len(matched_stems)} pairs, produced {len(diff_df)} comparison rows')
if errors:
    print(f'{len(errors)} pairs had errors:')
    for s, e in errors[:5]:
        print(f'  {os.path.basename(s)}: {e}')


Processed 53 pairs, produced 783 comparison rows
4 pairs had errors:
  all-images_across_targeted_structure_for_experience_level_barplot: missing reject column
  behavioral_across_targeted_structure_for_experience_level_barplot: missing reject column
  omissions_across_targeted_structure_for_experience_level_barplot: missing reject column
  task_across_targeted_structure_for_experience_level_barplot: missing reject column


## Summary

In [32]:
# Top-line: how many flipped?
counts = diff_df['status'].value_counts(dropna=False)
print('Comparison row counts by status:')
print(counts.to_string())

if 'flipped' in counts:
    n_flip = int(counts['flipped'])
    n_total = int(counts.get('same', 0) + n_flip)
    pct = 100 * n_flip / n_total if n_total else 0
    print(f'\nFlipped significance: {n_flip} of {n_total} comparisons ({pct:.1f}%)')

# By model_type — separates true MLM fits from auto-fallback (ANOVA/t-test) cases
if 'mlm_model_type' in diff_df.columns:
    print('\nFlips by underlying model used (mlm path):')
    print(diff_df[diff_df['status'] == 'flipped']['mlm_model_type'].value_counts(dropna=False).to_string())

Comparison row counts by status:
same             514
only_in_mlm      159
only_in_tukey     57
flipped           53

Flipped significance: 53 of 567 comparisons (9.3%)

Flips by underlying model used (mlm path):
anova    32
mlm      21


In [33]:
# Breakdown by figure stem — which figures have the most flips
if not diff_df.empty:
    per_figure = diff_df.groupby('figure_stem')['status'].value_counts().unstack(fill_value=0)
    per_figure['total'] = per_figure.sum(axis=1)
    per_figure = per_figure.sort_values('flipped' if 'flipped' in per_figure.columns else 'total',
                                        ascending=False)
    print('Per-figure flip counts (top 20):')
    print(per_figure.head(20).to_string())

Per-figure flip counts (top 20):
status                                                                                                                               flipped  only_in_mlm  only_in_tukey  same  total
figure_stem                                                                                                                                                                          
platform_paper_figures/figure_3_supplemental/response_metrics/experience_modulation_omissions                                             12            0              0    15     27
platform_paper_figures/figure_3_supplemental/metric_distributions/experience_modulation_omissions                                         12            0              0    15     27
platform_paper_figures/figure_4_supplemental/coding_scores_and_kernels/omissions_across_binned_depth_for_experience_level_barplot          5           18              0    31     54
platform_paper_figures/figure_3/metric_distributions/expe

In [34]:
# Show every flipped comparison with context
flipped = diff_df[diff_df['status'] == 'flipped'][:25]
print(f'\n--- First {len(flipped)} flipped comparisons ---')
cols_to_show = ['figure_stem', 'cell_type', 'group1', 'group2',
                'tukey_reject', 'mlm_reject',
                'mlm_pvalue_adj', 'mlm_omnibus_pvalue', 'mlm_icc', 'mlm_model_type', 'mlm_n_groups']
cols_to_show = [c for c in cols_to_show if c in flipped.columns]
print(flipped[cols_to_show].to_string(index=False))


--- First 25 flipped comparisons ---
                                                                                                                    figure_stem      cell_type         group1         group2 tukey_reject mlm_reject  mlm_pvalue_adj  mlm_omnibus_pvalue  mlm_icc mlm_model_type  mlm_n_groups
                                                     platform_paper_figures/figure_2/behavior_metrics/mean_dprime_engaged_stats            NaN       Familiar          Novel        False       True        0.010546        1.346529e-02 0.402426            mlm          63.0
                                                   platform_paper_figures/figure_2/behavior_metrics/mean_hit_rate_engaged_stats            NaN       Familiar        Novel +        False       True        0.026108        2.267819e-02 0.571320            mlm          65.0
                platform_paper_figures/figure_2_supplemental/behavior_metrics/mean_dprime_engaged_stats_sdk_mean_dprime_engaged            NaN       

In [35]:
# Show every flipped comparison with context
flipped = diff_df[diff_df['status'] == 'flipped'][20:]
print(f'\n--- Last {len(flipped)} flipped comparisons ---')
cols_to_show = ['figure_stem', 'cell_type', 'group1', 'group2',
                'tukey_reject', 'mlm_reject',
                'mlm_pvalue_adj', 'mlm_omnibus_pvalue', 'mlm_icc', 'mlm_model_type', 'mlm_n_groups']
cols_to_show = [c for c in cols_to_show if c in flipped.columns]
print(flipped[cols_to_show].to_string(index=False))


--- Last 33 flipped comparisons ---
                                                                                                                        figure_stem      cell_type         group1         group2 tukey_reject mlm_reject  mlm_pvalue_adj  mlm_omnibus_pvalue  mlm_icc mlm_model_type  mlm_n_groups
                                  platform_paper_figures/figure_3_supplemental/metric_distributions/experience_modulation_omissions            NaN     Excitatory Vip Inhibitory         True      False        0.604800        1.134387e-01      NaN          anova           NaN
                                  platform_paper_figures/figure_3_supplemental/metric_distributions/experience_modulation_omissions            NaN     Excitatory Vip Inhibitory         True      False        0.604800        1.134387e-01      NaN          anova           NaN
                                  platform_paper_figures/figure_3_supplemental/metric_distributions/experience_modulation_omissions       

In [36]:
# Save the full diff to CSV for the response-to-reviewers
diff_df.to_csv(output_csv, index=False)
print(f'\nWrote full diff to: {output_csv}')


Wrote full diff to: /Users/marinag/Library/CloudStorage/Dropbox/JupyterNotebooks/Figures/mlm_vs_tukey_diff_summary.csv


## How to read this

- **`same`**: Tukey and MLM agreed on significance. Most rows should be here.
- **`flipped`**: The two methods disagree. Worth checking against paper text.
- **`only_in_tukey` / `only_in_mlm`**: A row in one file has no counterpart in the other. Usually means the join keys (cell_type, comparison, etc.) differ between paths — check the raw CSVs if many of these appear.
- **`mlm_model_type='mlm'`**: The hierarchical model actually fit. The flip is meaningful — pseudoreplication was masking or inflating an effect.
- **`mlm_model_type='anova'`**: Auto-fallback fired (mouse-level data — typically behavior metrics after dropping duplicate sessions). Flips here reflect ANOVA vs. Welch's t-test differences, not MLM vs. ANOVA. Should be rare.

For the response-to-reviewers, the `flipped` rows are the ones worth discussing — they're the cases where accounting for nesting changed the conclusion.